# 🔧 02 — Transformación
**Metodología HEFESTO — Paso 3: Modelo Lógico del DW**

Lee los datos de `02_interim/`, aplica limpieza y construye las tablas del esquema Estrella:
`dimTiempo`, `dimProducto`, `dimGeografia`, `dimCanal` y `factVentas`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from src.transform import (
    cargar_config,
    limpiar_dataframe,
    construir_dim_tiempo,
    construir_dim_producto,
    construir_dim_geografia,
    construir_dim_canal,
    construir_fact_ventas,
    guardar_tablas_procesadas,
)

## 1️⃣ Leer datos intermedios

In [2]:
df_raw = pd.read_parquet("../data/02_interim/01_raw_cargado.parquet")
print(f"Filas cargadas: {len(df_raw):,}")
df_raw.head(3)

Filas cargadas: 128,975


,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,...,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,...,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,None,False,Easy Ship,None
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,...,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,None
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,...,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,None,None


## 2️⃣ Limpieza y normalización

In [3]:
config = cargar_config("../config/settings.yaml")
df_clean = limpiar_dataframe(df_raw, config)
df_clean.head(3)

[TRANSFORM] Iniciando limpieza...
[TRANSFORM] ✅ Limpieza completada. Shape: (128975, 22)
[TRANSFORM]    Nulos restantes: 7828


,order_id,fecha,status,tipo_fulfillment,canal_ventas,nivel_servicio,estilo,sku,categoria,talla,...,qty,moneda,amount,ciudad,estado,codigo_postal,pais,promociones,es_b2b,fulfilled_by
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,...,0,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,Sin Promocion,False,Easy Ship
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,...,1,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,...,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,No Informado


In [4]:
# Verificar que no queden nulos problemáticos
df_clean.isnull().sum()[df_clean.isnull().sum() > 0]

moneda           7795
codigo_postal      33
dtype: int64

## 3️⃣ Construir dimensiones (HEFESTO — Paso 3.2)

In [5]:
dim_tiempo = construir_dim_tiempo(df_clean)
print("\ndimTiempo:")
dim_tiempo

[TRANSFORM] ✅ dimTiempo: 91 filas

dimTiempo:


,idTiempo,fecha,dia,mes,nombre_mes,trimestre,anio
0,1,2022-03-31,31,3,March,1,2022
1,2,2022-04-01,1,4,April,2,2022
2,3,2022-04-02,2,4,April,2,2022
3,4,2022-04-03,3,4,April,2,2022
4,5,2022-04-04,4,4,April,2,2022
...,...,...,...,...,...,...,...
86,87,2022-06-25,25,6,June,2,2022
87,88,2022-06-26,26,6,June,2,2022
88,89,2022-06-27,27,6,June,2,2022
89,90,2022-06-28,28,6,June,2,2022


In [6]:
dim_producto = construir_dim_producto(df_clean)
print(f"\ndimProducto ({len(dim_producto)} filas):")
dim_producto.head(10)

[TRANSFORM] ✅ dimProducto: 7053 filas

dimProducto (7053 filas):


,idProducto,categoria,talla,estilo
0,1,Blouse,Free,BL003
1,2,Blouse,Free,BL017
2,3,Blouse,Free,BL001
3,4,Blouse,Free,BL026
4,5,Blouse,Free,BL057
5,6,Blouse,Free,BL053
6,7,Blouse,Free,BL013
7,8,Blouse,Free,BL020
8,9,Blouse,Free,BL021
9,10,Blouse,Free,BL009


In [7]:
dim_geografia = construir_dim_geografia(df_clean)
print(f"\ndimGeografia ({len(dim_geografia)} filas):")
dim_geografia.head(10)

[TRANSFORM] ✅ dimGeografia: 7401 filas

dimGeografia (7401 filas):


,idGeografia,ciudad,estado,pais
0,1,ANDAMAN AND NICOBAR ISLANDS PORT BLAIR,ANDAMAN & NICOBAR,IN
1,2,BAMBOOFLAT,ANDAMAN & NICOBAR,IN
2,3,FERRARGUNJ,ANDAMAN & NICOBAR,IN
3,4,GARACHARMA,ANDAMAN & NICOBAR,IN
4,5,GOALGHAR PORT BLAIR,ANDAMAN & NICOBAR,IN
5,6,GREAT NICOBAR,ANDAMAN & NICOBAR,IN
6,7,HADDO,ANDAMAN & NICOBAR,IN
7,8,HAVELOCK,ANDAMAN & NICOBAR,IN
8,9,JUNGLIGHAT,ANDAMAN & NICOBAR,IN
9,10,KAMORTA,ANDAMAN & NICOBAR,IN


In [8]:
dim_canal = construir_dim_canal(df_clean)
print("\ndimCanal:")
dim_canal

[TRANSFORM] ✅ dimCanal: 4 filas

dimCanal:


,idCanal,tipo_fulfillment,canal_ventas,nivel_servicio
0,1,Amazon,Amazon.in,Expedited
1,2,Amazon,Amazon.in,Standard
2,3,Amazon,Non-Amazon,Standard
3,4,Merchant,Amazon.in,Standard


## 4️⃣ Construir tabla de hechos (HEFESTO — Paso 3.3)

Indicadores:
- `cantidad_vendida`   = SUM(qty) donde status != Cancelled
- `monto_total`        = SUM(amount) donde status != Cancelled
- `cantidad_cancelada` = SUM(qty) donde status == Cancelled
- `monto_cancelado`    = SUM(amount) donde status == Cancelled

In [9]:
fact_ventas = construir_fact_ventas(
    df_clean, dim_tiempo, dim_producto, dim_geografia, dim_canal, config
)
fact_ventas.head(10)

[TRANSFORM] Construyendo factVentas...
[TRANSFORM] ✅ factVentas: 122,349 filas


,idProducto,idGeografia,idTiempo,idCanal,cantidad_vendida,monto_total,cantidad_cancelada,monto_cancelado
0,1,1947,19,4,1,419.0,0,0.00
1,1,2268,16,4,0,0.0,0,399.05
2,1,2472,18,4,1,419.0,0,0.00
3,1,3665,17,4,1,419.0,0,0.00
4,1,4219,18,1,1,419.0,0,0.00
5,1,4619,15,4,1,419.0,0,0.00
6,1,6007,28,1,0,0.0,0,0.00
7,1,6007,31,1,1,419.0,0,0.00
8,1,6238,2,4,1,412.0,0,0.00
9,1,6848,16,4,1,419.0,0,0.00


In [10]:
# Validación: totales deben coincidir con el CSV original
print("Total unidades vendidas:", fact_ventas["cantidad_vendida"].sum())
print(f"Total monto vendido (INR): {fact_ventas["monto_total"].sum():,.2f}")
print("Total unidades canceladas:", fact_ventas["cantidad_cancelada"].sum())

Total unidades vendidas: 110992
Total monto vendido (INR): 71,673,394.00
Total unidades canceladas: 5657


## 5️⃣ Guardar en `03_processed/`

In [11]:
guardar_tablas_procesadas(
    dim_tiempo, dim_producto, dim_geografia, dim_canal, fact_ventas, config
)
print("\n✅ Tablas guardadas en data/03_processed/")

[TRANSFORM] ✅ Guardado: data/03_processed/dimTiempo.parquet
[TRANSFORM] ✅ Guardado: data/03_processed/dimProducto.parquet
[TRANSFORM] ✅ Guardado: data/03_processed/dimGeografia.parquet
[TRANSFORM] ✅ Guardado: data/03_processed/dimCanal.parquet
[TRANSFORM] ✅ Guardado: data/03_processed/factVentas.parquet

✅ Tablas guardadas en data/03_processed/


## ✅ Resultado

Todas las tablas del DW fueron construidas y guardadas como `.parquet` en `03_processed/`.

▶️ Siguiente paso: corré `03_load.ipynb`